# Create Python 3.11 ENV

In [ ]:
!conda create -n femr-py311 python=3.11 -y
!conda run -n femr-py311 python -m pip install ipykernel
!conda run -n femr-py311 python -m pip install torch==2.1.2 \
    --index-url https://download.pytorch.org/whl/cu121
!conda run -n femr-py311 python -m ipykernel install \
    --user \
    --name femr-py311 \
    --display-name "Python 3.11 - FEMR"

In [ ]:
!conda run -n femr-py311 python -m pip install femr==0.2.3 datasets==2.15.0 xformers transformers==4.35.2


# Imports

In [ ]:
import os
import re
import pandas as pd
import numpy as np

try:
    from google.cloud import bigquery
except ImportError:
    bigquery = None

import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))

bucket = os.getenv("WORKSPACE_BUCKET")
cdr = os.environ.get("WORKSPACE_CDR")

if cdr is None:
    raise EnvironmentError(
        "WORKSPACE_CDR is not set. This script should be run inside an All of Us workspace."
    )

use_bqstorage = ("BIGQUERY_STORAGE_API_ENABLED" in os.environ)

# Load Model

In [ ]:
!conda run -n femr-py311 python -m pip install ipywidgets
from huggingface_hub import notebook_login
import femr.models.transformer
import torch
import femr.models.tokenizer
import femr.models.processor
import datetime

notebook_login()

In [ ]:
model_name = "StanfordShahLab/clmbr-t-base"

# Load tokenizer / batch loader
tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(model_name)
batch_processor = femr.models.processor.FEMRBatchProcessor(tokenizer)

# Load model
model = femr.models.transformer.FEMRModel.from_pretrained(model_name)

In [ ]:
import inspect

print(inspect.getsource(model.__class__))


# Load Data

In [ ]:
genetic_df = pd.read_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_full_v9.csv')
sample_meds = pd.read_parquet('/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort/data/train/0.parquet')
survey_df = pd.read_parquet('/home/jupyter/workspace/data_bucket/survey_data/survey.parquet')

In [ ]:
survey_df.head()

In [ ]:
survey_df.groupby(by='person_id').agg({'llm_event_text':' '.join})

In [ ]:
from pathlib import Path

dir_path = Path('/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort/data/train/')

for file in dir_path.glob('*.parquet'):
    meds = pd.read_parquet(file)
    
    